# **Trip Duration EDA**

##### By Sherif Ahmed

### Table of Contents :

- [Data Inspection](#data-inspection)
  - [How the data look like](#how-the-dataset-look-like)
  - [Summary Statistics](#summary-statistics)
- [Analysis features and Feature Engineering](#analysis-features-and-feature-engineering)
  - [Target Variable](#target-variable)
  - [Discrete numerical feature](#discrete-numerical-feature)
  - [Categorical Variables](#categorical-variable)
  - [Geographical Data](#geographical-data)
  - [Temporal/Time-date Analysis](#temporaltime-date-analysis)
- [Correlation Analysis](#correlation-analysis)
- [Modeling](#modeling)
- [Save the data](#save-the-model)


<a id="Data Inspection"></a>

# Data Inspection

Start by inspecting the dataset to get a general sense of its structure and contents.


In [1]:
# import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib.gridspec import GridSpec
from plotly.subplots import make_subplots

import seaborn as sns
from geopy import distance
import math
from geopy.point import Point


# Disable warnings 
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load the data and prepare it.

df1 = pd.read_csv("split/train.csv")
df2 = pd.read_csv("split/val.csv") 
df = pd.concat([df1, df2])

# Reset the index if needed
df.reset_index(drop=True, inplace=True)

In [ ]:
df1.shape , df2.shape

In [ ]:
df.shape 

In [ ]:
df.head()

In [ ]:
df.columns

<a id="sub1section1"></a>

#### How the dataset look like

We have 10 feathers and 1 target
let's go through each attribute briefly:

    id: A unique identifier for each trip. It serves as a primary key to distinguish one trip from another.

    vendor_id: A code indicating the provider associated with the trip record. This could represent different taxi companies or service providers.

    pickup_datetime: The date and time when the meter was engaged, marking the start of the trip.

    dropoff_datetime: The date and time when the meter was disengaged, marking the end of the trip.

    passenger_count: The number of passengers in the vehicle. It is a driver-entered value, indicating how many individuals were in the taxi during the trip.

    pickup_longitude: The longitude coordinate where the meter was engaged, i.e., the pickup location.

    engaged, i.e., the pickup location.

    dropoff_longitude: The longitude coordinate where the meter was disengaged, i.e., the dropoff location.

    dropoff_latitude: The latitude coordinate where the meter was disengaged, i.e., the dropoff location.

    store_and_fwd_flag: This flag indicates whether the data was sent to the vendor in real-time ("N") or whether it was stored in the vehicle's memory and sent later when a connection was available ("Y").

    trip_duration: The duration of the trip in seconds, i.e., the time between the pickup and dropoff.

Note : trip_duration is our target Variable.


<a id="Sub2section1"></a>

# Summary Statistics


In [ ]:
df.describe().T

- There are two vendor / taxi companies is there difference speed in each one.
- Haveing Nine passengers is challenging.
- zero passengers ?.
- The max Trip duration took 3.526282e+06 in sec or 58771 minutes is approximately 40 days and 1111 minutes so its definitely outlier ?.


In [ ]:
num_duplicates = df.duplicated().sum()
print(f"Number of NaN values: {df.isna().sum().sum()}")
print(f"Number of Null values: {df.isnull().sum().sum()}")
print(f"Number of duplicates: {num_duplicates}")

In [ ]:
def get_featurs_details(feature_name:str):
    print(df[feature_name].describe(),end="\n\n")
    print(df[feature_name].value_counts(),end="\n\n")

In [ ]:
get_featurs_details(feature_name="vendor_id")

- Its look like the vendor_id is Categorical Variable so its means we have two vendor or service.


In [ ]:
get_featurs_details(feature_name="passenger_count")

- Its look like the passenger_count also a Categorical Variable so its means the range of peaple who cant taxi draveled is between [1,8].

- The min number of passenger is 0 its definitely a noise (Maybe it happens because error in the system or the driver forget to enter the value)


In [ ]:
get_featurs_details(feature_name="pickup_longitude")

In [ ]:
get_featurs_details(feature_name="pickup_latitude")

In [ ]:
get_featurs_details(feature_name="dropoff_longitude")

In [ ]:
get_featurs_details(feature_name="dropoff_latitude")

- In the last four feature looking at them individually make it not Interesting to focus on them.
- I will try combining them to get more in knowledge it [Geographical data analysis](#geographical-data) section.


In [ ]:
get_featurs_details(feature_name="store_and_fwd_flag")

### Analyze date and time.


In [ ]:
get_featurs_details(feature_name="pickup_datetime")

In [ ]:
get_featurs_details(feature_name="dropoff_datetime")

- The Time and date format does help us to get information or knowledge about different time snippets or Months or even days affecting the taxi trip duration.

- I will try to fix in [Temporaltime date analysis](#temporaltime-date-analysis) section.


<a id="Analysis features"></a>

# Analysis features and Feature Engineering


<a id="Target Variable"> </a>

## Target Variable


In [ ]:
print(df["trip_duration"].describe(),end='\n\n') # The Summary statistics of trip duration by secound is hard to measured / understand.

# We will try to convert in minutes to make the numbers more easy to read.
trip_by_seconds = df['trip_duration'].copy()
trip_by_minutes =  trip_by_seconds / 60 # Convert seconds to minutes
trip_by_minutes = pd.DataFrame(trip_by_minutes.round().astype(int))  # Convert to integer

print('std   ', trip_by_minutes.std())
print('min   ', trip_by_minutes.min())
print('mean  ', trip_by_minutes.mean())
print('median', trip_by_minutes.median())
print('max   ', trip_by_minutes.max())


- The max Trip duration took 58771 minutes is approximately 40 days and 1111 minutes so definitely outlier.
- The mean is maybe effected little bit because of ourliers so we use the median to detect where the center of data.
- Looks like the most of Trip duration takes a about approximation 11 minutes
- The min Trip duration took 0 minutes seems to be outlier.


In [ ]:
df['trip_by_seconds_Log_Transformed'] = np.log1p(df['trip_duration'].values) # we preform the lop1p transfrom to can Visualize better.
fig = sns.histplot(df['trip_by_seconds_Log_Transformed'] ,label = "Trip duration")
plt.title('Distribution of Trip Duration by seconds (Log Transformed)')
plt.xlabel('Trip duration')
plt.ylabel('Frqency')
plt.show()

- The Target Variable Distribution look like Gaussian distribution.

- The two sides of Distribution make clear to us the outliers in dataset.


In [ ]:
df.drop(columns=['id'],axis=1,inplace=True) # drop id Variable.

<a id="Discrete Numerical Feature"></a>

## Discrete Numerical Feature


In [ ]:
#Creating a list of all our categorical variables
cat = ["vendor_id","passenger_count"] # From before, we know that this feature can be treated like Categorical Variables
for col in cat:
    print("{} has {} unique values.".format(col,df[col].unique()))

In [ ]:
# Assuming 'trip_log' and 'df' are defined and ready to use

# Create the figure with GridSpec and specify the size
fig = plt.figure(constrained_layout=True, figsize=(12, 10))
gs = GridSpec(2, 2, figure=fig)

# Subplot 1: Bar Plot for vendor_id
ax1 = fig.add_subplot(gs[0, 0])
sns.barplot(data=df, x="vendor_id", y= df['trip_by_seconds_Log_Transformed'], palette='hot', ax=ax1)
ax1.set_title('Trip Duration vs Vendor id Barplot')
ax1.set_xlabel("Vendor id", fontsize=12)
ax1.set_ylabel("Trip duration", fontsize=12)

# Subplot 2: Bar Plot for passenger_count
ax2 = fig.add_subplot(gs[0, 1])
sns.barplot(data=df, x=df["passenger_count"], y= df['trip_by_seconds_Log_Transformed'], palette='magma', ax=ax2)
ax2.set_title('Trip Duration vs Passenger count Barplot')
ax2.set_xlabel("Passenger count", fontsize=12)
ax2.set_ylabel("Trip duration", fontsize=12)

# Add a title for the entire figure
fig.suptitle("Discrete Numerical Analysis", fontsize=16)

# Show the plots
plt.show()

- Type of Vendor Not effect much with trip duration.

- When the number of passenger groups from [1 to 6] take constant trip duration and the number of passenger groups from [7 to 8] take less trip duration I think its becauce:

  - Vehicle Capacity -> It's possible that the vehicles used by both vendors have a maximum capacity of 6 passengers. When there are 6 or fewer passengers, the vehicles are operating at their maximum capacity, and the trip duration remains constant because the vehicles are fully utilized.

  - Vehicle Type -> It's also possible that the two vehicle vendors have different types of vehicles in their fleet. One vendor might have larger vehicles capable of accommodating more passengers, while the other vendor might have smaller vehicles. The larger vehicles can comfortably accommodate 7 to 8 passengers, resulting in shorter trip durations.

  - Or the passengers groups from [7 to 8] just travel less than another groups.

- If **vendor of the taxi not effect with trip duration so idea of vehicle type and vehicle capacity not correct** so we need to use a boxplot
  to detect If we just dealing with some random noise or passengers groups from [7 to 8] just travel less than another groups.


In [ ]:
fig = plt.figure(constrained_layout=True, figsize=(12, 10))
gs = GridSpec(1, 2, figure=fig)

# Subplot 1: Box Plot for vendor_id
ax1 = fig.add_subplot(gs[0, 0])
sns.boxplot(data=df, x="vendor_id", y= df['trip_by_seconds_Log_Transformed'], palette='hot', ax=ax1)
ax1.set_title("Trip Duration vs Passenger Count Boxplot")
ax1.set_xlabel("Vendor id", fontsize=12)
ax1.set_ylabel("Trip duration", fontsize=12)

# Subplot 2: Box Plot for passenger_count
ax2 = fig.add_subplot(gs[0, 1])
sns.boxplot(data=df, x="passenger_count", y= df['trip_by_seconds_Log_Transformed'], palette='magma', ax=ax2)
ax2.set_title("Trip Duration vs Passenger Count Boxplot")
ax2.set_xlabel("Passenger count", fontsize=12)
ax2.set_ylabel("Trip duration", fontsize=12)

# Add a title for the entire figure
fig.suptitle("Discrete Numerical Analysis", fontsize=16)

# Show the plots
plt.show()

- Now it's clear to us vehicle type Not effect much with trip duration and number of passenger groups from [1 to 6] take constant trip duration.

- my conclusion passengers groups from [7 to 8] just travel less than another groups.


<a id="Categorical variable"></a>

## Categorical variable


In [ ]:
for col in df.columns: 
    if df[col].dtype == "object" and (col not in ["pickup_datetime","dropoff_datetime"]):
        cat.append(col)

print("Categorical variables :: \n\n{}".format(cat)) 

In [ ]:
fig = plt.figure(constrained_layout=True, figsize=(12, 10))
gs = fig.add_gridspec(1, 1)  

ax = fig.add_subplot(gs[0, 0])  # Add a single subplot
sns.histplot(data=df, x='store_and_fwd_flag', y=df['trip_by_seconds_Log_Transformed'], palette='magma', ax=ax)
ax.set_title('Trip Duration vs S&f flag Boxplot')
ax.set_xlabel('store and fwd flag', fontsize=12)
ax.set_ylabel('Trip duration', fontsize=12)

plt.show()

- The most of txai trips sent to the vendor in real-time ("N").

- The most of trips who sent to the vendor in real-time ("N") likey take more Trip duration.


<a id="Geographical Data"></a>

## Geographical Data

Now lets to analysis Latitude and longitude are geographical coordinates


In [ ]:
Geo_df = pd.DataFrame()

def haversine_distance(df):
    pick = (df['pickup_latitude'], df['pickup_longitude'])
    drop = (df['dropoff_latitude'], df['dropoff_longitude'])
    dist = distance.geodesic(pick, drop).km
    return dist

# Geo_df['distance'] = df.apply(haversine_distance, axis=1) # get the totall distance from Latitude and longitude coordinates


In [ ]:
fig = plt.figure(constrained_layout=True, figsize=(12, 10))
gs = fig.add_gridspec(1, 1)  
ax = fig.add_subplot(gs[0, 0])  # Add a single subplot

distance_Suitable = Geo_df['distance'] >= 1
sns.histplot(Geo_df['distance'][distance_Suitable])

sns.histplot(data=Geo_df, x='distance', palette='magma', ax=ax)
ax.set_title('Distance')
ax.set_xlabel('Distance in kilometer', fontsize=12)
plt.xlim(0, 40)
plt.show()

- Looks like most of trip goes from less then 1 kile meter to 25 kilometer


In [ ]:
Geo_df['speed_kmh']= Geo_df['distance']/(df['trip_duration']/(60*60)) # Speed = distance / time

# Create the histogram plot

speed_high = Geo_df['speed_kmh'] >= 1
sns.histplot(Geo_df['speed_kmh'][speed_high])

# Set the x-axis limit to the desired range
plt.xlim(0, 150)
# Add labels and title (optional)
plt.xlabel('Speed (km/h)')
plt.ylabel('Frequency')
plt.title('Distribution of Speed')
plt.show()

- The most of trip goes speed in range 1-40 Km/h.


<a id="Temporal/Time-date Analysis"></a>

## Temporal/Time-date Analysis


In [ ]:
# Correct formatting date and time.

df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])

df["dropoff_datetime"] = pd.to_datetime(df["dropoff_datetime"])

In [ ]:
# Now from every data/time we can get a new information like Months/day/ Morning or afternoon or night / season for each trip
# Example of extracting additional information from the "pickup_datetime" column
df_date_time = pd.DataFrame()        

df_date_time["pickup_month"] = df["pickup_datetime"].dt.month
df_date_time["pickup_day"] = df["pickup_datetime"].dt.day
df_date_time["pickup_hour"] = df["pickup_datetime"].dt.hour
df_date_time["pickup_dayofweek"] = df["pickup_datetime"].dt.dayofweek
df_date_time["pickup_quarter"] = df["pickup_datetime"].dt.quarter
df_date_time["duration_in_seconds"] = (df["dropoff_datetime"] - df["pickup_datetime"]).dt.total_seconds() #/ 3600
df_date_time["duration_in_seconds_log"] = np.log1p(df_date_time["duration_in_seconds"])

bins = [0, 2, 5, 8, 11, 12]  # 0, 2, 5, 8, 11, 12 represent the starting and ending months of each season
labels = ['Winter', 'Spring', 'Summer', 'Autumn', 'Winter']  # Labels for each season


def get_time_period(hour):
    if 5 <= hour < 12:   # Morning: 5 AM to 11:59 AM
        return "Morning"
    elif 12 <= hour < 17:   # Afternoon: 12 PM to 4:59 PM
        return "Afternoon"
    else:   # Night: 5 PM to 4:59 AM (next day)
        return "Night"

# Create a new column 'Season' using pd.cut()
df_date_time['pickup_Season'] = pd.cut(df_date_time["pickup_month"] , bins=bins, labels=labels, right=False,ordered=False) 
# This Colums just to make it Easier to understand the relationship with weather seasons witha trip duration

df_date_time["time_period"] = df_date_time["pickup_hour"].apply(get_time_period)

day_names = ['Saturday', 'Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
df_date_time['pickup_dayofweek_ch'] = df_date_time['pickup_dayofweek'].map(lambda x: day_names[x])

df_date_time.head(5)

### Now lets try to see Relationship between Trip duration vs Trip_Season & Trip timeperiod


In [ ]:
# Create the figure with GridSpec and specify the size
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 8))

# Subplot 1: Bar Plot for Seasons
sns.barplot(data=df_date_time, x="pickup_Season", y=df['trip_by_seconds_Log_Transformed'], palette='hot', ax=ax1)
ax1.set_title('Trip Duration vs Seasons Barplot')
ax1.set_xlabel("Seasons", fontsize=12)
ax1.set_ylabel("Trip Duration", fontsize=12)

# Subplot 2: Bar Plot for time period
sns.barplot(data=df_date_time, x="time_period", y=df['trip_by_seconds_Log_Transformed'], palette='magma', ax=ax2)
ax2.set_title('Trip Duration vs Trip Time Period Barplot')
ax2.set_xlabel("Trip Time Period", fontsize=12)
ax2.set_ylabel("Trip Duration", fontsize=12)

# Subplot 3: Bar Plot for months
sns.barplot(data=df_date_time, x="pickup_month", y=df['trip_by_seconds_Log_Transformed'], palette='magma', ax=ax3)
ax3.set_title('Trip Duration vs Months Barplot')
ax3.set_xlabel("Months", fontsize=12)
ax3.set_ylabel("Trip Duration", fontsize=12)

# Adjust layout for better appearance
plt.tight_layout()

# Show the plots
plt.show()

- The longer trip durations during summer might be attributed to vacations and holidays, which lead to increased traffic on the roads.
- Longer trip durations in the afternoon can be explained by higher levels of crowding during that time of day.
- April, May, and July experience longer trip durations compared to other months.


In [ ]:

# Create the figure with GridSpec and specify the size
fig = plt.figure(constrained_layout=True, figsize=(12, 10))
gs = GridSpec(2, 2, figure=fig)

# Subplot 1: Bar Plot for Days of the week
ax1 = fig.add_subplot(gs[0, 0])
sns.barplot(data=df_date_time, x="pickup_dayofweek_ch", y= df['trip_by_seconds_Log_Transformed'], palette='hot', ax=ax1)
ax1.set_title('Trip Duration vs Days of the week Barplot')
ax1.set_xlabel("Days of the week", fontsize=12)
ax1.set_ylabel("Trip duration", fontsize=12)

# Subplot 2: Bar Plot for quarter of the year
ax2 = fig.add_subplot(gs[0, 1])
sns.barplot(data=df_date_time, x="pickup_quarter", y= df['trip_by_seconds_Log_Transformed'], palette='magma', ax=ax2)
ax2.set_title('Trip Duration vs The quarter of the year Barplot')
ax2.set_xlabel("The quarter of the year", fontsize=12)
ax2.set_ylabel("Trip duration", fontsize=12)

# Subplot 3: Bar Plot for Each day of the month
ax3 = fig.add_subplot(gs[1, :])
sns.barplot(data=df_date_time, x="pickup_day", y= df['trip_by_seconds_Log_Transformed'], palette='magma', ax=ax3)
ax3.set_title('Trip Duration vs Each day Barplot')
ax3.set_xlabel("Each day of the month", fontsize=12)
ax3.set_ylabel("Trip duration", fontsize=12)

# Add a title for the entire figure
fig.suptitle("Temporal Analysis", fontsize=16)

# Show the plots
plt.show()

- Looks like Monday,Tuesday and Wenesday experience longer trip durations compared to other months.
- The other distributions looks normal.


<a id="Correlation Analysis"></a>

# Correlation Analysis


In [ ]:
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.show()

- There posstive relation trip duration (in Seconds log-Transformed) with pickup longitude ,dropoff longitude and passenger count.

- There negative relation trip duration (in Seconds log-Transformed) with pickup latitude and dropoff latitude.


In [ ]:
df_date_time["trip_by_seconds_Log_Transformed"] = df["trip_by_seconds_Log_Transformed"]
correlation_matrix = df_date_time.corr()

sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.show()

In [ ]:
Geo_df["trip_by_seconds_Log_Transformed"] = df["trip_by_seconds_Log_Transformed"]
correlation_matrix = Geo_df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.show()

- There strong posstive relation trip duration (in Seconds log-Transformed) with distance.

- There negative relation trip duration (in Seconds log-Transformed) with speed kmh.


Conclusion we have a pretty Good Relationship between the distance features so we can use it and modeling.


<a id="Modeling"></a>

# Modeling


Lets perpar our detset


In [2]:
df_train = pd.read_csv('split/train.csv')
df_test = pd.read_csv('split/val.csv')

# df_train = pd.concat([df_train, df_val])
# # Reset the index if needed
# df_train.reset_index(drop=True, inplace=True)

# df_test = pd.read_csv('split_sample/test.csv')

In [3]:
t_train = df_train['trip_duration'].to_numpy().reshape(-1,1)
t_test = df_test['trip_duration'].to_numpy().reshape(-1,1)

In [4]:
# transfrom trip_duration to log_trip_duration_sec
t_train = np.log1p(t_train)
t_test = np.log1p(t_test)

In [6]:

df_train["pickup_datetime"] = pd.to_datetime(df_train["pickup_datetime"]) 
df_test["pickup_datetime"] = pd.to_datetime(df_test["pickup_datetime"]) 


In [7]:
# From our EDA, we can use these features.

df_train["pickup_hour"] = df_train["pickup_datetime"].dt.hour
df_train["pickup_day"] = df_train["pickup_datetime"].dt.day
df_train["pickup_dayofweek"] = df_train["pickup_datetime"].dt.dayofweek
df_train["pickup_month"] = df_train["pickup_datetime"].dt.month

df_test["pickup_hour"] = df_test["pickup_datetime"].dt.hour
df_test["pickup_day"] = df_test["pickup_datetime"].dt.day
df_test["pickup_dayofweek"] = df_test["pickup_datetime"].dt.dayofweek
df_test["pickup_month"] = df_test["pickup_datetime"].dt.month

In [8]:
def haversine_distance_km(df):
    pick = Point(df['pickup_latitude'], df['pickup_longitude'])
    drop = Point(df['dropoff_latitude'], df['dropoff_longitude'])
    dist = distance.geodesic(pick, drop).miles
    return dist

In [9]:
def haversine_distance_miles(df):
    pick = Point(df['pickup_latitude'], df['pickup_longitude'])
    drop = Point(df['dropoff_latitude'], df['dropoff_longitude'])
    dist = distance.geodesic(pick, drop).miles
    return dist

In [10]:
def haversine_distance_meters(df):
    pick = Point(df['pickup_latitude'], df['pickup_longitude'])
    drop = Point(df['dropoff_latitude'],df['dropoff_longitude'])
    dist = (distance.geodesic(pick, drop).meters)
    return dist

In [11]:
def calculate_direction(row):
    pickup_coordinates =  Point(row['pickup_latitude'], row['pickup_longitude'])
    dropoff_coordinates = Point(row['dropoff_latitude'], row['dropoff_longitude'])
    
    # Calculate the difference in longitudes
    delta_longitude = dropoff_coordinates[1] - pickup_coordinates[1]
    
    # Calculate the bearing (direction) using trigonometry
    y = math.sin(math.radians(delta_longitude)) * math.cos(math.radians(dropoff_coordinates[0]))
    x = math.cos(math.radians(pickup_coordinates[0])) * math.sin(math.radians(dropoff_coordinates[0])) - \
        math.sin(math.radians(pickup_coordinates[0])) * math.cos(math.radians(dropoff_coordinates[0])) * \
        math.cos(math.radians(delta_longitude))
    
    # Calculate the bearing in degrees
    bearing = math.atan2(y, x)
    bearing = math.degrees(bearing)
    
    # Adjust the bearing to be in the range [0, 360)
    bearing = (bearing + 360) % 360
    
    return bearing


In [ ]:
# def haversine_distance_centimeters(df):
#     pick = (df['pickup_latitude'], df['pickup_longitude'])
#     drop = (df['dropoff_latitude'],df['dropoff_longitude'])
#     dist = (distance.geodesic(pick, drop).meters * 100)
#     return dist

In [12]:
# From our EDA, can use these distance features.
df_train['distance_km'] = df_train.apply(haversine_distance_km, axis=1) 
df_test['distance_km'] = df_test.apply(haversine_distance_km, axis=1)   

df_train['distance_miles'] = df_train.apply(haversine_distance_miles, axis=1) 
df_test['distance_miles'] = df_test.apply(haversine_distance_miles, axis=1)  
 
df_train['distance_meters'] = df_train.apply(haversine_distance_meters, axis=1) 
df_test['distance_meters'] = df_test.apply(haversine_distance_meters, axis=1)  

df_train['direction'] = df_train.apply(calculate_direction, axis=1)
df_test['direction'] = df_test.apply(calculate_direction  , axis=1)


# df_train['distance_cm'] = df_train.apply(haversine_distance_centimeters, axis=1) 
# df_test['distance_cm'] = df_test.apply(haversine_distance_centimeters, axis=1)   


: 

: 

In [ ]:
encoding  = pd.get_dummies(df_train['store_and_fwd_flag'])
df_train = pd.concat([df_train, encoding], axis=1)

encoding  = pd.get_dummies(df_test['store_and_fwd_flag'])
df_test = pd.concat([df_test, encoding], axis=1)

In [ ]:
df_train = df_train.drop(columns=['id', 'pickup_datetime', 'dropoff_datetime','store_and_fwd_flag','trip_duration'] ,axis=1)

df_test = df_test.drop(columns=['id', 'pickup_datetime', 'dropoff_datetime','store_and_fwd_flag','trip_duration'] ,axis=1)

In [ ]:
df_train.columns

Index(['vendor_id', 'passenger_count', 'pickup_longitude', 'pickup_latitude',
       'dropoff_longitude', 'dropoff_latitude', 'pickup_hour', 'pickup_day',
       'pickup_dayofweek', 'pickup_month', 'distance_km', 'distance_miles',
       'distance_meters', 'direction', 'N', 'Y'],
      dtype='object')

In [123]:
# df_train.to_csv("split/model_train.csv")
# df_test.to_csv("split/model_test.csv")

- Save the dataframe after Preparing to the modeling phase.


In [5]:
df_train = pd.read_csv('split/model_train.csv')
df_test = pd.read_csv('split/model_test.csv')

In [6]:
df_train['distance_km'] = round(np.log1p(df_train['distance_km']))
df_test['distance_km'] = round(np.log1p(df_test['distance_km']))

In [7]:
df_train['distance_meters'] = round(np.log1p(df_train['distance_meters']))
df_test['distance_meters'] = round(np.log1p(df_test['distance_meters']))

In [8]:
df_train['distance_miles'] = round(np.log1p(df_train['distance_miles']))
df_test['distance_miles'] = round(np.log1p(df_test['distance_miles']))

In [9]:
df_train['direction'] = round(np.log1p(df_train['direction']))
df_test['direction'] = round(np.log1p(df_test['direction']))

In [10]:
df_train['pickup_latitude'] = round(np.log1p(df_train['pickup_latitude']))
df_test['pickup_latitude'] = round(np.log1p(df_test['pickup_latitude']))

# df_train['pickup_longitude'] = round(np.log1p(np.abs(df_train['pickup_longitude'])))
# df_test['pickup_longitude'] = round(np.log1p(np.abs(df_test['pickup_longitude'])))

# df_train['dropoff_longitude'] = round(np.log1p(np.abs(df_train['dropoff_longitude'])))
# df_test['dropoff_longitude'] = round(np.log1p(np.abs(df_test['dropoff_longitude'])))

df_train['dropoff_latitude'] = round(np.log1p(df_train['dropoff_latitude']))
df_test['dropoff_latitude'] = round(np.log1p(df_test['dropoff_latitude']))

In [11]:
df_train = df_train.drop(columns=['Unnamed: 0','vendor_id','passenger_count'],axis=1)
df_test = df_test.drop(columns=['Unnamed: 0','vendor_id','passenger_count'],axis=1)
df_train.head()

,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,pickup_hour,pickup_day,pickup_dayofweek,pickup_month,distance_km,distance_miles,distance_meters,direction,N,Y
0,-73.985611,4.0,-73.980331,4.0,7,8,2,6,1.0,1.0,8.0,2.0,1,0
1,-73.978394,4.0,-73.991623,4.0,12,3,6,4,1.0,1.0,8.0,5.0,1,0
2,-73.989059,4.0,-73.973381,4.0,2,5,6,6,1.0,1.0,7.0,4.0,1,0
3,-73.990326,4.0,-73.991264,4.0,17,5,3,5,1.0,1.0,8.0,6.0,1,0
4,-73.789497,4.0,-73.987137,4.0,17,12,3,5,3.0,3.0,10.0,6.0,1,0


In [12]:
df_train.columns

Index(['pickup_longitude', 'pickup_latitude', 'dropoff_longitude',
       'dropoff_latitude', 'pickup_hour', 'pickup_day', 'pickup_dayofweek',
       'pickup_month', 'distance_km', 'distance_miles', 'distance_meters',
       'direction', 'N', 'Y'],
      dtype='object')

In [13]:
x_train = df_train.iloc[:, :-1].to_numpy()
x_test = df_test.iloc[:, :-1].to_numpy()

In [14]:
# import Libraries for modeling.
from sklearn.preprocessing import MinMaxScaler,StandardScaler,PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score,mean_squared_error

Splitting and preprocessing the data


In [16]:
def data_preprocessing(x_train: np.ndarray, x_val: np.ndarray, t_train: np.ndarray, t_val: np.ndarray, choice: int = 1):
    if choice == 1:
        processor = MinMaxScaler() 
    else:
        processor = StandardScaler()
    x_train = processor.fit_transform(x_train)
    x_val = processor.transform(x_val)
    return x_train, x_val, t_train, t_val

In [17]:
def monomials_poly_features(X:np.ndarray, degree: int):
    new_x = X.copy()
    for col in X.T:
        for d in range(2, degree + 1, 1):
            transformed_col = np.power(col, d)
            new_x = np.insert(new_x, -1, transformed_col, axis=1)
    return new_x

In [18]:
x_train = monomials_poly_features(x_train,26)
x_test = monomials_poly_features(x_test,26)

In [88]:
# poly = PolynomialFeatures(degree=3,include_bias=True, interaction_only=True)
# x_train = poly.fit_transform(x_train)
# x_test = poly.transform(x_test)

In [19]:
x_train.shape

(1000000, 338)

In [20]:
x_train = x_train.astype('int')
x_test = x_test.astype('int')  

In [21]:
x_train, x_test, t_train, t_test = data_preprocessing(x_train, x_test, t_train, t_test, 1) 

In [22]:
model = Ridge(alpha=1) 
model.fit(x_train, t_train)

Ridge(alpha=1)

In [23]:
t_train_pred = model.predict(x_train)
acc = r2_score(t_train,y_pred=t_train_pred)
error = mean_squared_error(t_train,y_pred=t_train_pred)
print(f"Model Accuracy r2_socer in matrix :{acc}")
print(f"Model train error :{error}")

Model Accuracy r2_socer in matrix :0.6349467870028664
Model train error :0.23057378895536343


In [24]:
t_val_pred = model.predict(x_test)    
acc = r2_score(t_test,y_pred=t_val_pred)
error = mean_squared_error(t_test,y_pred=t_val_pred)
print(f"Model Accuracy r2_socer in matrix :{acc}")
print(f"Model vaildtion error :{error}")
model.coef_

Model Accuracy r2_socer in matrix :0.6335121827575443
Model vaildtion error :0.23456562148346508


array([[-3.87227300e+00,  0.00000000e+00,  4.73448694e+00,
         0.00000000e+00,  9.65307392e-01, -2.98764492e-01,
         7.04818416e-01,  2.88591585e-02,  1.26195583e+00,
         3.35516200e+00, -1.05609100e+01, -3.07891385e+00,
         9.14192847e+00, -4.75504057e+00,  9.31305133e-01,
         1.59158097e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+0

<a id="Save the model"></a>

# Save the model


In [ ]:
# import joblib
# filename = 'baseline_model_0.67_0.50__.pkl'
# joblib.dump(model, filename)